# 05 - Preparar dados para modelagem e comparar normalizacoes

Este notebook repete o fluxo do notebook 40: carrega os Parquets intermediarios, seleciona descricoes, remove nulos e vazios, normaliza texto, consolida duplicatas preservando frequencia, agrega metadados e valida o resultado.

Tambem cria colunas experimentais sem substituir `descricao_normalizada`: uma padroniza abreviacoes e unidades; outra uniformiza a separacao entre numeros e unidades; e uma terceira mascara valores numericos. As colunas experimentais sao alternativas de entrada para os proximos experimentos de TF-IDF e podem conter descricoes repetidas, por isso nao alteram a chave de consolidacao nem a frequencia. A gravacao final usa um arquivo separado e falha se ele ja existir.

In [1]:
from __future__ import annotations

import sys
import re
import unicodedata
import os
from pathlib import Path

import pandas as pd

for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    source_dir = path / "src"
    if source_dir.exists():
        sys.path.append(str(source_dir))
        break
else:
    raise FileNotFoundError("Nao encontrei a pasta src nos diretorios pais.")

from nofis_classifier.paths import find_project_root

## Configuracao

Defina `MANTER_DUPLICATAS = True` para gerar uma linha por item valido, sem consolidar descricoes normalizadas. Com `False`, o fluxo consolida descricoes e registra a frequencia, como antes. Cada modo usa um nome de saida proprio.

As regras experimentais abaixo sao deliberadamente conservadoras. Revise os exemplos e compare cada coluna em experimentos separados antes de escolher uma representacao para o modelo.

In [2]:
DATASET_NAME = "2025_NCMs_filtrados"
MANTER_DUPLICATAS = False

EXPERIMENT_DATASET_NAME = (f"{DATASET_NAME}-sem-duplicatas")
FREQUENCY_COLUMN = "frequencia"
NUMERIC_SEQUENCE_PATTERN = r"\d{6,}" # remover sequencias de dígitos das descrições

PROJECT_ROOT = find_project_root()
INTERIM_DIR = PROJECT_ROOT / "data" / "interim" / DATASET_NAME
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / EXPERIMENT_DATASET_NAME

# Mapeamentos testados na coluna descricao_abreviacoes_unidades.
ABBREVIATION_MAP = {
    "und": "un",
    "unidade": "un",
    "unidades": "un",
    "quilo": "kg",
    "quilos": "kg",
    "quilograma": "kg",
    "quilogramas": "kg",
    "grama": "g",
    "gramas": "g",
    "litro": "l",
    "litros": "l",
    "lt": "l",
    "mililitro": "ml",
    "mililitros": "ml",
    "pct": "pacote",
    "c": "com",
    "s": "sem",
}

INTERIM_DIR, PROCESSED_DIR

(WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/interim/2025_NCMs_filtrados'),
 WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/2025_NCMs_filtrados-sem-duplicatas'))

## Funcoes de preparacao

In [3]:
def normalize_column_name(column_name: str) -> str:
    """Normalize a column name enough to detect expected fields."""
    text = unicodedata.normalize("NFKD", str(column_name))
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = text.upper()
    text = re.sub(r"[^A-Z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def find_column(columns: pd.Index, required_tokens: tuple[str, ...]) -> str:
    """Find the first column whose normalized name contains all required tokens."""
    for column in columns:
        normalized = normalize_column_name(column)
        if all(token in normalized for token in required_tokens):
            return column
    raise KeyError(f"Nao encontrei coluna com tokens {required_tokens}. Colunas: {list(columns)}")


def remove_long_numeric_sequences(value: object) -> str:
    """Remove digit sequences with six or more digits from a description."""
    if pd.isna(value):
        return ""

    text = re.sub(NUMERIC_SEQUENCE_PATTERN, " ", str(value))
    return re.sub(r"\s+", " ", text).strip()


def normalize_description(value: object) -> str:
    """Normalize product descriptions for TF-IDF and duplicate consolidation."""
    if pd.isna(value):
        return ""

    text = str(value).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def compact_month(period: str) -> str:
    """Convert YYYY-MM to YYYYMM for artifact names."""
    return period.replace("-", "")


def most_frequent_by_description(
    frame: pd.DataFrame,
    value_column: str,
    output_column: str,
) -> pd.DataFrame:
    """Return the most common value per normalized description."""
    valid = frame[["descricao_normalizada", value_column]].dropna(subset=[value_column]).copy()
    valid = valid[valid[value_column].astype(str).str.strip() != ""]
    if valid.empty:
        return pd.DataFrame(columns=["descricao_normalizada", output_column])

    weighted = (
        valid.groupby(["descricao_normalizada", value_column], as_index=False)
        .size()
        .sort_values(["descricao_normalizada", "size"], ascending=[True, False])
    )
    return weighted.drop_duplicates("descricao_normalizada")[["descricao_normalizada", value_column]].rename(
        columns={value_column: output_column}
    )

In [4]:
def normalize_abbreviations_and_units(value: object) -> str:
    """Standardize a small, explicit set of abbreviations and unit names."""
    if pd.isna(value):
        return ""

    # Expand slash abbreviations before punctuation normalization removes `/`.
    raw_text = re.sub(r"(?i)\bc\s*/", "com ", str(value))
    raw_text = re.sub(r"(?i)\bs\s*/", "sem ", raw_text)
    text = normalize_description(raw_text)
    for source, replacement in ABBREVIATION_MAP.items():
        text = re.sub(rf"\b{re.escape(source)}\b", replacement, text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_number_unit_spacing(value: object) -> str:
    """Make number/unit spacing consistent while retaining numeric values."""
    text = normalize_description(value)
    # `500ml`, `500 ml` -> `500 ml`; same for common units.
    text = re.sub(r"(?<=\d)(kg|mg|g|ml|l|un)\b", r" \1", text)
    return re.sub(r"\s+", " ", text).strip()


def mask_numeric_values(value: object) -> str:
    """Replace numeric values with NUMERO while retaining unit tokens."""
    text = normalize_number_unit_spacing(value)
    text = re.sub(r"\b\d+(?:\s+\d+)*\b", " numero ", text)
    return re.sub(r"\s+", " ", text).strip()


## Testes rapidos da normalizacao

In [5]:
normalization_examples = {
    " ARROZ TIPO 1 5KG ": "arroz tipo 1 5kg",
    "CAFE TORRADO 500g": "cafe torrado 500g",
    "LEITE UHT 1L": "leite uht 1l",
    "ACUCAR-CRISTAL 2 KG": "acucar cristal 2 kg",
}

for raw, expected in normalization_examples.items():
    normalized = normalize_description(raw)
    assert normalized == expected, (raw, normalized, expected)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

informative_examples = ["QUEIJO MUCARELA, KG", "Couve Flor (Hortifruti) Ceasa", "PIMENTAO VERMELHA CEASA"]
for raw in informative_examples:
    normalized = normalize_description(raw)
    assert len(normalized) >= 3, (raw, normalized)
    assert any(char.isalpha() for char in normalized), (raw, normalized)

duplicate_check = pd.DataFrame(
    {
        "descricao_original": ["Arroz Tipo 1", " arroz tipo 1 ", "ARROZ TIPO 1"],
    }
)
duplicate_check["descricao_normalizada"] = duplicate_check["descricao_original"].map(normalize_description)
duplicate_frequency = duplicate_check.groupby("descricao_normalizada").size().iloc[0]
assert duplicate_frequency == len(duplicate_check)

numeric_sequence_examples = {
    "ARROZ COD123456789 TIPO 1": "ARROZ COD TIPO 1",
    "LEITE 12345 1L": "LEITE 12345 1L",
    "ITEM 123456 E LOTE 987654321": "ITEM E LOTE",
}
for raw, expected in numeric_sequence_examples.items():
    actual = remove_long_numeric_sequences(raw)
    assert actual == expected, (raw, actual, expected)

sequence_check = pd.DataFrame({"descricao_original": ["ARROZ COD123456 TIPO 1"]})
sequence_check["descricao_original"] = sequence_check["descricao_original"].map(
    remove_long_numeric_sequences
)
assert len(sequence_check) == 1
assert sequence_check.loc[0, "descricao_original"] == "ARROZ COD TIPO 1"


## Variantes experimentais de texto

`descricao_normalizada` permanece como baseline. As variantes abaixo usam regras deterministicas e deixam quantidades disponiveis em colunas alternativas: abreviacoes/unidades padronizadas, espacamento numero-unidade canonico e numeros substituidos por `numero`. `c` e `s` sao expandidos apenas como tokens isolados, portanto inspecione exemplos reais antes de interpretar o impacto.

In [6]:
variant_examples = {
    "ARROZ 1KG": ("arroz 1kg", "arroz 1 kg", "arroz numero kg"),
    "ARROZ 1 KG": ("arroz 1 kg", "arroz 1 kg", "arroz numero kg"),
    "LEITE 500ML": ("leite 500ml", "leite 500 ml", "leite numero ml"),
    "LEITE 500 ML": ("leite 500 ml", "leite 500 ml", "leite numero ml"),
    "CAFE 250 GRAMAS": ("cafe 250 gramas", "cafe 250 gramas", "cafe numero gramas"),
}
for raw, expected in variant_examples.items():
    actual = (
        normalize_description(raw),
        normalize_number_unit_spacing(raw),
        mask_numeric_values(raw),
    )
    assert actual == expected, (raw, actual, expected)

assert normalize_abbreviations_and_units("LEITE 1 LITRO") == "leite 1 l"
assert normalize_abbreviations_and_units("SABAO 2 UND") == "sabao 2 un"
assert normalize_abbreviations_and_units("CAFE C/ ACUCAR") == "cafe com acucar"



## Carregar Parquets intermediarios

In [7]:
parquet_files = sorted(INTERIM_DIR.glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError(f"Nenhum Parquet encontrado em {INTERIM_DIR}")

frames = []
for parquet_file in parquet_files:
    frame = pd.read_parquet(parquet_file)
    frame["competencia"] = parquet_file.stem
    frames.append(frame)

df = pd.concat(frames, ignore_index=True)

print(f"Arquivos carregados: {len(parquet_files)}")
print(f"Linhas intermediarias: {len(df):,}")
df.head()

Arquivos carregados: 12
Linhas intermediarias: 868,292


,CHAVE DE ACESSO,MODELO,SÉRIE,NÚMERO,NATUREZA DA OPERAÇÃO,DATA EMISSÃO,EVENTO MAIS RECENTE,DATA/HORA EVENTO MAIS RECENTE,CPF/CNPJ Emitente,RAZÃO SOCIAL EMITENTE,...,NÚMERO PRODUTO,DESCRIÇÃO DO PRODUTO/SERVIÇO,CÓDIGO NCM/SH,NCM/SH (TIPO DE PRODUTO),CFOP,QUANTIDADE,UNIDADE,VALOR UNITÁRIO,VALOR TOTAL,competencia
0,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,2.0,ACHOCOLATADO 400G,18069000.0,Outros chocolates e preparações alimentícias c...,5102.0,"110,00",UNIDAD,"5,80","638,00",2025-01
1,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,4.0,"APRESUNTADO, TIPO CARNE SUINA",16010000.0,"Enchidos e produtos semelhantes, de carne, de ...",5102.0,"40,00",KG,"17,50","700,00",2025-01
2,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,5.0,BARRAS CEREAIS FRUTAS SECAS E MEL EMB 24 UND,17049090.0,"Outros produtos de confeitaria, sem cacau",5102.0,"50,00",EMB,"20,00","1000,00",2025-01
3,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,6.0,BISCOITO DOCE COM RECHEIO PCT 78G,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante",5102.0,"39,00",PCT,"1,78","69,42",2025-01
4,22250119864509000100550010000025281104500492,55 - NF-E EMITIDA EM SUBSTITUIÇÃO AO MODELO 1 ...,1,2528,VENDA DE MERCADORIA,01/01/2025 08:41:00,NaN,NaN,19864509000100,NASCIMENTO SALES PRODUTOS ALIMENTICIOS LTDA EPP,...,7.0,BISCOITO MAIZENA 400G,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante",5102.0,"150,00",PCT,"3,85","577,50",2025-01


## Selecionar descricoes

In [8]:
description_col = find_column(df.columns, ("DESCRI", "PRODUTO"))

support_columns = {
    "codigo_ncm": None,
    "ncm_tipo": None,
}

try:
    support_columns["codigo_ncm"] = find_column(df.columns, ("NCM", "SH"))
except KeyError:
    pass

try:
    support_columns["ncm_tipo"] = find_column(df.columns, ("TIPO", "PRODUTO"))
except KeyError:
    pass

selected_columns = [description_col, "competencia"]
selected_columns.extend(column for column in support_columns.values() if column is not None)

rename_columns = {description_col: "descricao_original"}
if support_columns["codigo_ncm"] is not None:
    rename_columns[support_columns["codigo_ncm"]] = "codigo_ncm"
if support_columns["ncm_tipo"] is not None:
    rename_columns[support_columns["ncm_tipo"]] = "ncm_tipo"

df_selected = df[selected_columns].rename(columns=rename_columns)

df_selected.head()

,descricao_original,competencia,codigo_ncm,ncm_tipo
0,ACHOCOLATADO 400G,2025-01,18069000.0,Outros chocolates e preparações alimentícias c...
1,"APRESUNTADO, TIPO CARNE SUINA",2025-01,16010000.0,"Enchidos e produtos semelhantes, de carne, de ..."
2,BARRAS CEREAIS FRUTAS SECAS E MEL EMB 24 UND,2025-01,17049090.0,"Outros produtos de confeitaria, sem cacau"
3,BISCOITO DOCE COM RECHEIO PCT 78G,2025-01,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante"
4,BISCOITO MAIZENA 400G,2025-01,19053100.0,"Bolachas e biscoitos, adicionados de edulcorante"


## Remover sequencias numericas longas, normalizar e consolidar duplicatas

Cadeias com seis ou mais digitos consecutivos sao removidas do texto da descricao antes da normalizacao. A linha e preservada quando restam outros termos; descricoes que ficam vazias continuam sujeitas ao filtro existente de descricoes invalidas.

In [9]:
input_rows = len(df_selected)

df_valid = df_selected.dropna(subset=["descricao_original"]).copy()
df_valid["descricao_original"] = df_valid["descricao_original"].astype(str).str.strip()
df_valid = df_valid[df_valid["descricao_original"] != ""].copy()

numeric_sequences_removed = int(
    df_valid["descricao_original"].str.count(NUMERIC_SEQUENCE_PATTERN).sum()
)
df_valid["descricao_original"] = df_valid["descricao_original"].map(
    remove_long_numeric_sequences
)

df_valid["descricao_normalizada"] = df_valid["descricao_original"].map(normalize_description)
df_valid = df_valid[df_valid["descricao_normalizada"] != ""].copy()

# Variantes para comparar em experimentos; a consolidacao segue o baseline.
df_valid["descricao_abreviacoes_unidades"] = df_valid["descricao_original"].map(
    normalize_abbreviations_and_units
)
df_valid["descricao_numero_unidade_padronizada"] = df_valid["descricao_original"].map(
    normalize_number_unit_spacing
)
df_valid["descricao_numeros_mascarados"] = df_valid["descricao_original"].map(mask_numeric_values)

valid_rows = len(df_valid)

print(f"Linhas com coluna de descricao: {input_rows:,}")
print(f"Linhas com descricao valida: {valid_rows:,}")
print(f"Sequencias numericas removidas: {numeric_sequences_removed:,}")
print(f"Linhas removidas por nulos/vazios: {input_rows - valid_rows:,}")

Linhas com coluna de descricao: 868,292
Linhas com descricao valida: 868,288
Sequencias numericas removidas: 28,213
Linhas removidas por nulos/vazios: 4


In [10]:
if MANTER_DUPLICATAS:
    df_modeling = df_valid.copy()
    df_modeling[FREQUENCY_COLUMN] = 1
    df_modeling["descricao_original_exemplo"] = df_modeling["descricao_original"]
    df_modeling["descricoes_originais_distintas"] = 1
    df_modeling["primeiro_mes"] = df_modeling["competencia"]
    df_modeling["ultimo_mes"] = df_modeling["competencia"]
    df_modeling["meses_presentes"] = 1
    rename_columns = {
        "codigo_ncm": "codigo_ncm_mais_frequente",
        "ncm_tipo": "ncm_tipo_mais_frequente",
    }
    df_modeling = df_modeling.rename(
        columns={key: value for key, value in rename_columns.items() if key in df_modeling.columns}
    )
    df_modeling = df_modeling.sort_values(
        ["competencia", "descricao_normalizada"], ascending=[True, True]
    ).reset_index(drop=True)
else:
    aggregations = {
        FREQUENCY_COLUMN: ("descricao_original", "size"),
        "descricao_original_exemplo": ("descricao_original", "first"),
        "descricoes_originais_distintas": ("descricao_original", "nunique"),
        "primeiro_mes": ("competencia", "min"),
        "ultimo_mes": ("competencia", "max"),
        "meses_presentes": ("competencia", "nunique"),
        "descricao_abreviacoes_unidades": ("descricao_abreviacoes_unidades", "first"),
        "descricao_numero_unidade_padronizada": ("descricao_numero_unidade_padronizada", "first"),
        "descricao_numeros_mascarados": ("descricao_numeros_mascarados", "first"),
    }

    df_modeling = (
        df_valid.groupby("descricao_normalizada", as_index=False)
        .agg(**aggregations)
        .sort_values([FREQUENCY_COLUMN, "descricao_normalizada"], ascending=[False, True])
        .reset_index(drop=True)
    )

    if "codigo_ncm" in df_valid.columns:
        ncm_mode = most_frequent_by_description(df_valid, "codigo_ncm", "codigo_ncm_mais_frequente")
        df_modeling = df_modeling.merge(ncm_mode, on="descricao_normalizada", how="left")

    if "ncm_tipo" in df_valid.columns:
        ncm_tipo_mode = most_frequent_by_description(df_valid, "ncm_tipo", "ncm_tipo_mais_frequente")
        df_modeling = df_modeling.merge(ncm_tipo_mode, on="descricao_normalizada", how="left")

df_modeling.head(10)


,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,papel a4 br go office 75g 500fl certfscmisto70...,7814,Papel A4 BR Go Office 75g 500fl CertFSCMisto70...,1,2025-01,2025-12,12,papel a4 br go office 75g 500fl certfscmisto70...,papel a4 br go office 75 g 500fl certfscmisto7...,papel a4 br go office numero g 500fl certfscmi...,48025610.0,"Outros papéis e cartões,sem fibras obtidas por..."
1,pilha alcalina go tech aaa lr03 palito 2un,3588,Pilha Alcalina Go Tech AAA LR03 Palito 2UN,1,2025-01,2025-12,12,pilha alcalina go tech aaa lr03 palito 2un,pilha alcalina go tech aaa lr03 palito 2 un,pilha alcalina go tech aaa lr03 palito numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"
2,farinha de mandioca,3334,FARINHA DE MANDIOCA,10,2025-01,2025-12,12,farinha de mandioca,farinha de mandioca,farinha de mandioca,11062000.0,"Farinhas, sêmolas e pós, de sagu ou das raízes..."
3,papel a4 75g 500fls autopaper fsc credito mist...,3115,PAPEL A4 75G 500FLS AUTOPAPER FSC CREDITO MIST...,1,2025-03,2025-12,10,papel a4 75g 500fls autopaper fsc credito mist...,papel a4 75 g 500fls autopaper fsc credito mis...,papel a4 numero g 500fls autopaper fsc credito...,48025610.0,"Outros papéis e cartões,sem fibras obtidas por..."
4,pilha alcalina go tech aa lr6 pequena 2un,3032,Pilha Alcalina Go Tech AA LR6 Pequena 2UN,1,2025-01,2025-12,12,pilha alcalina go tech aa lr6 pequena 2un,pilha alcalina go tech aa lr6 pequena 2 un,pilha alcalina go tech aa lr6 pequena numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"
5,20 litros sem gas agua mineral,2272,20 LITROS SEM GAS - AGUA MINERAL,1,2025-01,2025-12,12,20 l sem gas agua mineral,20 litros sem gas agua mineral,numero litros sem gas agua mineral,22011000.0,"Águas minerais e águas gaseificadas, não adici..."
6,esponja dupla face,2220,ESPONJA DUPLA FACE,6,2025-01,2025-12,12,esponja dupla face,esponja dupla face,esponja dupla face,68053090.0,Outros abrasivos naturais/artificiais em pó/gr...
7,linguica calabresa,2192,LINGUICA CALABRESA,7,2025-01,2025-12,12,linguica calabresa,linguica calabresa,linguica calabresa,16010000.0,"Enchidos e produtos semelhantes, de carne, de ..."
8,pilha alcalina aaa c 2un,2179,PILHA ALCALINA AAA C/2UN.,2,2025-01,2025-12,12,pilha alcalina aaa com 2un,pilha alcalina aaa c 2 un,pilha alcalina aaa c numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"
9,pao frances,2121,PAO FRANCES,13,2025-01,2025-12,12,pao frances,pao frances,pao frances,19059090.0,"Outros produtos de padaria, pastelaria, indúst..."


## Validacoes finais

In [11]:
assert not df_modeling.empty
assert df_modeling["descricao_normalizada"].notna().all()
assert (df_modeling["descricao_normalizada"].str.strip() != "").all()
assert int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows
if not MANTER_DUPLICATAS:
    assert df_modeling["descricao_normalizada"].is_unique
else:
    assert len(df_modeling) == valid_rows
    assert df_modeling[FREQUENCY_COLUMN].eq(1).all()

print(f"Linhas na base: {len(df_modeling):,}")
print(f"Descricoes unicas normalizadas: {df_modeling['descricao_normalizada'].nunique():,}")
print(f"Duplicatas mantidas: {MANTER_DUPLICATAS}")
print(f"Linhas validas consolidadas: {int(df_modeling[FREQUENCY_COLUMN].sum()):,}")
print(f"Frequencia preservada: {int(df_modeling[FREQUENCY_COLUMN].sum()) == valid_rows}")

Linhas na base: 149,149
Descricoes unicas normalizadas: 149,149
Duplicatas mantidas: False
Linhas validas consolidadas: 868,288
Frequencia preservada: True


## Salvar base processada

A saida e gravada em `data/processed/` na pasta definida por `EXPERIMENT_DATASET_NAME`, separada da base do notebook 40. Com duplicatas mantidas, cada linha representa um item valido e frequencia vale 1. No modo consolidado, as colunas alternativas representam a primeira descricao de cada grupo baseline.

In [12]:
start_period = compact_month(min(file.stem for file in parquet_files))
end_period = compact_month(max(file.stem for file in parquet_files))

# Salva o Parquet
output_path = PROCESSED_DIR / f"{EXPERIMENT_DATASET_NAME}-{start_period}-{end_period}.parquet"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if output_path.exists():
    raise FileExistsError(
        f"Arquivo processado ja existe: {output_path}. "
        "Escolha outro nome ou remova o arquivo manualmente se quiser regenerar."
    )
df_modeling.to_parquet(output_path, index=False)

# Salva o CSV
output_path = PROCESSED_DIR / f"{EXPERIMENT_DATASET_NAME}-{start_period}-{end_period}.csv"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
if output_path.exists():
    raise FileExistsError(
        f"Arquivo processado ja existe: {output_path}. "
        "Escolha outro nome ou remova o arquivo manualmente se quiser regenerar."
    )
df_modeling.to_csv(output_path, index=False)

output_path

WindowsPath('C:/Users/samue/Scripts/Nofis-Classifier/data/processed/2025_NCMs_filtrados-sem-duplicatas/2025_NCMs_filtrados-sem-duplicatas-202501-202512.csv')

In [13]:
df_modeling.head()

,descricao_normalizada,frequencia,descricao_original_exemplo,descricoes_originais_distintas,primeiro_mes,ultimo_mes,meses_presentes,descricao_abreviacoes_unidades,descricao_numero_unidade_padronizada,descricao_numeros_mascarados,codigo_ncm_mais_frequente,ncm_tipo_mais_frequente
0,papel a4 br go office 75g 500fl certfscmisto70...,7814,Papel A4 BR Go Office 75g 500fl CertFSCMisto70...,1,2025-01,2025-12,12,papel a4 br go office 75g 500fl certfscmisto70...,papel a4 br go office 75 g 500fl certfscmisto7...,papel a4 br go office numero g 500fl certfscmi...,48025610.0,"Outros papéis e cartões,sem fibras obtidas por..."
1,pilha alcalina go tech aaa lr03 palito 2un,3588,Pilha Alcalina Go Tech AAA LR03 Palito 2UN,1,2025-01,2025-12,12,pilha alcalina go tech aaa lr03 palito 2un,pilha alcalina go tech aaa lr03 palito 2 un,pilha alcalina go tech aaa lr03 palito numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"
2,farinha de mandioca,3334,FARINHA DE MANDIOCA,10,2025-01,2025-12,12,farinha de mandioca,farinha de mandioca,farinha de mandioca,11062000.0,"Farinhas, sêmolas e pós, de sagu ou das raízes..."
3,papel a4 75g 500fls autopaper fsc credito mist...,3115,PAPEL A4 75G 500FLS AUTOPAPER FSC CREDITO MIST...,1,2025-03,2025-12,10,papel a4 75g 500fls autopaper fsc credito mist...,papel a4 75 g 500fls autopaper fsc credito mis...,papel a4 numero g 500fls autopaper fsc credito...,48025610.0,"Outros papéis e cartões,sem fibras obtidas por..."
4,pilha alcalina go tech aa lr6 pequena 2un,3032,Pilha Alcalina Go Tech AA LR6 Pequena 2UN,1,2025-01,2025-12,12,pilha alcalina go tech aa lr6 pequena 2un,pilha alcalina go tech aa lr6 pequena 2 un,pilha alcalina go tech aa lr6 pequena numero un,85061019.0,"Outras pilhas alcalinas, de dióxido de manganês"


## Analise breve do dataset final

In [14]:
analysis_summary = pd.Series(
    {
        "arquivos_origem": len(parquet_files),
        "linhas_originais": len(df),
        "linhas_com_descricao_valida": valid_rows,
        "descricoes_unicas_normalizadas": len(df_modeling),
        "frequencia_total_preservada": int(df_modeling[FREQUENCY_COLUMN].sum()),
        "frequencia_media_por_descricao": df_modeling[FREQUENCY_COLUMN].mean(),
        "frequencia_mediana_por_descricao": df_modeling[FREQUENCY_COLUMN].median(),
        "maior_frequencia": int(df_modeling[FREQUENCY_COLUMN].max()),
        "periodo_inicial": df_modeling["primeiro_mes"].min(),
        "periodo_final": df_modeling["ultimo_mes"].max(),
    }
)

analysis_summary

arquivos_origem                           12
linhas_originais                      868292
linhas_com_descricao_valida           868288
descricoes_unicas_normalizadas        149149
frequencia_total_preservada           868288
frequencia_media_por_descricao      5.821615
frequencia_mediana_por_descricao         1.0
maior_frequencia                        7814
periodo_inicial                      2025-01
periodo_final                        2025-12
dtype: object

In [15]:
variant_collision_summary = pd.Series({
    "descricoes_baseline_unicas": df_modeling["descricao_normalizada"].nunique(),
    "unicas_abreviacoes_unidades": df_modeling["descricao_abreviacoes_unidades"].nunique(),
    "unicas_numero_unidade_padronizada": df_modeling["descricao_numero_unidade_padronizada"].nunique(),
    "unicas_numeros_mascarados": df_modeling["descricao_numeros_mascarados"].nunique(),
})
variant_collision_summary

descricoes_baseline_unicas           149149
unicas_abreviacoes_unidades          148522
unicas_numero_unidade_padronizada    147969
unicas_numeros_mascarados            138048
dtype: int64

In [16]:
frequency_bins = pd.cut(
    df_modeling[FREQUENCY_COLUMN],
    bins=[0, 1, 2, 5, 10, 50, 100, float("inf")],
    labels=["1", "2", "3-5", "6-10", "11-50", "51-100", "101+"],
)

frequency_profile = (
    frequency_bins.value_counts(sort=False)
    .rename_axis("faixa_frequencia")
    .reset_index(name="descricoes")
)
frequency_profile["percentual"] = (
    frequency_profile["descricoes"] / len(df_modeling) * 100
).round(2)

frequency_profile

,faixa_frequencia,descricoes,percentual
0,1,83992,56.31
1,2,22700,15.22
2,3-5,20635,13.84
3,6-10,9853,6.61
4,11-50,9707,6.51
5,51-100,1251,0.84
6,101+,1011,0.68


In [17]:
top_descriptions = df_modeling[
    ["descricao_normalizada", "descricao_original_exemplo", FREQUENCY_COLUMN, "meses_presentes"]
].head(20)

top_descriptions

,descricao_normalizada,descricao_original_exemplo,frequencia,meses_presentes
0,papel a4 br go office 75g 500fl certfscmisto70...,Papel A4 BR Go Office 75g 500fl CertFSCMisto70...,7814,12
1,pilha alcalina go tech aaa lr03 palito 2un,Pilha Alcalina Go Tech AAA LR03 Palito 2UN,3588,12
2,farinha de mandioca,FARINHA DE MANDIOCA,3334,12
3,papel a4 75g 500fls autopaper fsc credito mist...,PAPEL A4 75G 500FLS AUTOPAPER FSC CREDITO MIST...,3115,10
4,pilha alcalina go tech aa lr6 pequena 2un,Pilha Alcalina Go Tech AA LR6 Pequena 2UN,3032,12
5,20 litros sem gas agua mineral,20 LITROS SEM GAS - AGUA MINERAL,2272,12
6,esponja dupla face,ESPONJA DUPLA FACE,2220,12
7,linguica calabresa,LINGUICA CALABRESA,2192,12
8,pilha alcalina aaa c 2un,PILHA ALCALINA AAA C/2UN.,2179,12
9,pao frances,PAO FRANCES,2121,12
